# Seaquest — Actor lambda-sweep + on-trajectory preview (Colab GPU)
We have only ever deployed the **greedy argmax-critic** policy (success_by_H=0.533 full-view). The original Contrastive RL (Eysenbach et al. 2022) deploys a **learned actor** pi(a|s,g) that maximises the critic, with an optional BC term — we never ran it. This trains that actor **faithfully** (reuses `train_actor.train_actor()` verbatim: objective `(1-lam)*E_pi[f(s,a,g)] + lam*log pi(a_teacher) + ent_coef*H(pi)`, critic frozen) and sweeps `lam in {0 pure-critic, 0.5 critic+BC, 1 pure-BC}`.

**Entropy bonus (`--ent-coef`, default 0.01).** The discrete actor needs an explicit `H(pi)` term, else `lam=0` collapses the softmax to a constant action on this near-flat critic (action-score range ~0.16).

**Gradient balancing (`--balance-grad`).** Even after per-state z-scoring, the critic term's gradient is ~9x weaker than the BC term's here, so a nominal `lam=0.5` is really ~9:1 BC. `--balance-grad` per-step equalizes the critic-term gradient norm to the BC term's, so `lam` genuinely interpolates BC<->critic. **Use it** — otherwise lam=0.5 ≈ pure BC and the critic's contribution is untestable.

**Actor trains in FP32** (AMP only for the frozen-critic forwards; fp16 policy-gradients corrupted training).

Env-free preview; closed-loop success is a separate env step. **Needs:** GitHub TOKEN + Drive `raw_hf.zip` + Drive `critic_full_view.pt`.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
# 1. Clone repo at the committed actor-sweep script (main, latest).
TOKEN = 'PASTE_YOUR_GITHUB_TOKEN_HERE'
import os, subprocess
D = '/content/Goal-Conditioned-Confounded-MsPacman'
if not os.path.isdir(D):
    subprocess.run(['git','clone',f'https://{TOKEN}@github.com/tingrui-huang/Goal-Conditioned-Confounded-MsPacman.git',D], check=True)
%cd /content/Goal-Conditioned-Confounded-MsPacman
!git pull -q && git log --oneline -1
assert os.path.exists('seaquest_ccrl/scripts/g0_actor_lambda_sweep.py'), 'sweep script missing — pull main'

In [ ]:
# 2. raw_hf from Drive (mount + unzip + auto-locate traj_*.npz).
import glob, os, zipfile
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/raw_hf.zip'         # <-- EDIT if elsewhere
assert os.path.exists(ZIP), f'zip not found at {ZIP}'
EXTRACT = '/content/raw_hf_extracted'
if not glob.glob(EXTRACT + '/**/traj_0000.npz', recursive=True):
    with zipfile.ZipFile(ZIP) as z: z.extractall(EXTRACT)
DATA_ROOT = os.path.dirname(glob.glob(EXTRACT + '/**/traj_0000.npz', recursive=True)[0])
n = len(glob.glob(DATA_ROOT + '/traj_*.npz'))
print('DATA_ROOT =', DATA_ROOT, '| trajectories:', n); assert n == 40

In [ ]:
# 3. Locate the PRE-TRAINED full-view (oracle) critic on Drive (auto-search MyDrive).
import glob
CRITIC = '/content/drive/MyDrive/critic_full_view.pt'    # <-- EDIT if elsewhere
if not os.path.exists(CRITIC):
    hits = glob.glob('/content/drive/MyDrive/**/critic_full_view.pt', recursive=True)
    assert hits, 'critic_full_view.pt not found on Drive — upload it (from artifacts/seaquest/seaquest_stage_g0_fullview_train/)'
    CRITIC = hits[0]
import torch
c = torch.load(CRITIC, map_location='cpu', weights_only=False)
print('CRITIC =', CRITIC, '| oracle =', c['oracle'], '| frame_stack =', c['cfg'].get('frame_stack'))
assert c['oracle'] is True and c['cfg'].get('frame_stack') == 4, 'expected the 4-frame full-view/oracle critic'

In [ ]:
# 4. SMOKE (one lambda, 20 steps) — verify the pipeline + JSON shape before the ~1.5h run.
!python -m seaquest_ccrl.scripts.g0_actor_lambda_sweep --critic-ckpt "$CRITIC" --root "$DATA_ROOT" --lams 0.5 --steps 20 --balance-grad --out-dir /content/actor_smoke
import json; print(json.dumps(json.load(open('/content/actor_smoke/on_trajectory_preview.json'))['lambda_sweep'], indent=2))

In [ ]:
# 5. FULL sweep: lam in {0,0.5,1} x 20k, entropy 0.01, GRADIENT-BALANCED (so lam truly interpolates).
#    SANITY (watch the log): BC-acc must climb past chance (1/18=0.056); 'H' should NOT crash to 0.
#    With --balance-grad, lam=0.5 should differ from lam=1 (teacher agreement lower, vs-critic higher);
#    if lam=0.5 STILL == lam=1, the critic genuinely adds nothing (clean A). Also run WITHOUT the flag
#    once for the contrast (the ~9:1-BC baseline).
!python -u -m seaquest_ccrl.scripts.g0_actor_lambda_sweep --critic-ckpt "$CRITIC" --root "$DATA_ROOT" --lams 0.0,0.5,1.0 --steps 20000 --ent-coef 0.01 --balance-grad --out-dir /content/actor_lambda_sweep_balanced

In [ ]:
# 6. Read the preview. Judge collapse by ENTROPY (ln18=2.89 uniform), not by argmax share.
#    Key contrast: does BALANCED lam=0.5 diverge from lam=1 (pure BC)?
#      lam=0.5 top1_teacher << lam=1, and top1_vs_argmaxC > chance  -> critic pulls (was scale-drowned).
#      lam=0.5 ~ lam=1 even balanced                                -> critic genuinely useless (clean A).
import json
r = json.load(open('/content/actor_lambda_sweep_balanced/on_trajectory_preview.json'))
print(f"critic={r['view']} steps={r['steps']} ent_coef={r['ent_coef']} balance_grad={r.get('balance_grad')}  (ln18=2.89)\n")
hdr = ['lam','top1_teach','vdir_teach','top1_vsCrit','entropy','collapsed','val_actor','val_argmax']
print('{:>5} {:>11} {:>11} {:>12} {:>8} {:>10} {:>10} {:>10}'.format(*hdr))
for lam, d in r['lambda_sweep'].items():
    print('{:>5} {:>11.3f} {:>11.3f} {:>12.3f} {:>8.3f} {:>10} {:>10.3f} {:>10.3f}'.format(
        lam, d['top1_agree_teacher'], d['vdir_agree_teacher'], d['top1_agree_argmax_critic'],
        d['actor_entropy_nats'], str(d['entropy_collapsed']), d['critic_value_actor_Epi_f'],
        d['critic_value_argmax_f']))

In [ ]:
# 7. Persist preview JSON + the 3 actor checkpoints to Drive + download zip.
import shutil, glob, os
OUT = '/content/actor_sweep_out'; os.makedirs(OUT, exist_ok=True)
shutil.copy('/content/actor_lambda_sweep_balanced/on_trajectory_preview.json', OUT)
for p in glob.glob('/content/actor_lambda_sweep_balanced/lam*/actor_oracle.pt'):
    lam = os.path.basename(os.path.dirname(p))            # lam0.00 / lam0.50 / lam1.00
    shutil.copy(p, os.path.join(OUT, f'actor_oracle_{lam}.pt'))
shutil.make_archive('seaquest_actor_lambda_sweep_balanced', 'zip', OUT)
try:
    from google.colab import drive; drive.mount('/content/drive')
    shutil.copy('seaquest_actor_lambda_sweep_balanced.zip', '/content/drive/MyDrive/')
    print('copied zip to Drive')
except Exception as e:
    print('drive copy skipped:', e)
from google.colab import files; files.download('seaquest_actor_lambda_sweep_balanced.zip')

## Next — closed-loop verdict (env)
This preview is **env-free**. The actual B/C/D answer is the closed-loop success-rate: run `g0_closed_loop_eval.py` with each `actor_oracle_lam*.pt` (`--actor-ckpt`) on the **same anchor protocol** as the 0.533 argmax baseline, and compare success_by_H / success_at_H across argmax-critic vs lam=0/0.5/1.0 vs random. Paste `on_trajectory_preview.json` back for the preview read first.